# IS 362 – Week 5 Assignment  
**Name:** Sam Sikder  
**Course:** IS 362  
**Date:** February 28, 2026

In [1]:
import pandas as pd

AIRPORTS_URL = "https://raw.githubusercontent.com/hadley/nycflights13/master/data-raw/airports.csv"
WEATHER_URL  = "https://raw.githubusercontent.com/hadley/nycflights13/master/data-raw/weather.csv"

airports = pd.read_csv(AIRPORTS_URL)
weather  = pd.read_csv(WEATHER_URL)

airports.head()

,faa,name,lat,lon,alt,tz,dst,tzone
0,04G,Lansdowne Airport,41.130472,-80.619583,1044,-5,A,America/New_York
1,06A,Moton Field Municipal Airport,32.460572,-85.680028,264,-6,A,America/Chicago
2,06C,Schaumburg Regional,41.989341,-88.101243,801,-6,A,America/Chicago
3,06N,Randall Airport,41.431912,-74.391561,523,-5,A,America/New_York
4,09J,Jekyll Island Airport,31.074472,-81.427778,11,-5,A,America/New_York


# Question 1: Northernmost Airport in the U.S.

In [3]:
# Filter to likely U.S. airports (longitude should be negative)
us_airports = airports[airports["lon"] < 0]

# Now sort by latitude
north5 = us_airports.sort_values("lat", ascending=False).head(5)

north5[["faa", "name", "lat", "lon", "tzone"]]

,faa,name,lat,lon,tzone
230,BRW,Wiley Post Will Rogers Mem,71.285446,-156.766003,America/Anchorage
110,AIN,Wainwright Airport,70.638056,-159.994722,America/Anchorage
708,K03,Wainwright As,70.613378,-159.860350,America/Anchorage
152,ATK,Atqasuk Edward Burnell Sr Memorial Airport,70.467300,-157.436000,America/Anchorage
1363,UUK,Ugnu-Kuparuk Airport,70.330833,-149.597500,America/Anchorage


### Answer:

After filtering to U.S. airports (longitude < 0), the northernmost airport in the United States is **Wiley Post Will Rogers Memorial Airport (BRW)** located in Barrow, Alaska.

The airport has a latitude of 71.285446, making it the highest latitude (most northern) airport in the dataset.

### Validation & Assumptions:

Initially, the dataset returned Dillant Hopkins Airport as the northernmost airport. However, its longitude was positive (42.89), placing it outside the United States. To correct this, I filtered the dataset to include only airports with negative longitude values, which represent locations west of the Prime Meridian.

After applying this filter, Wiley Post Will Rogers Memorial Airport in Alaska was confirmed as the northernmost U.S. airport. External geographic references confirm that Barrow (Utqiaġvik), Alaska is the northernmost city in the United States.

# Question 2: Easternmost Airport in the U.S.

In [6]:
# Start with U.S. airports (longitude < 0)
us_airports = airports[airports["lon"] < 0]

# Easternmost = highest longitude (least negative)
east5 = us_airports.sort_values("lon", ascending=False).head(5)

east5[["faa", "name", "lat", "lon", "tzone"]]

,faa,name,lat,lon,tzone
444,EPM,Eastport Municipal Airport,44.910111,-67.012694,America/New_York
624,HUL,Houlton Intl,46.123083,-67.792056,America/New_York
259,CAR,Caribou Muni,46.871500,-68.017917,America/New_York
1101,PQI,Northern Maine Rgnl At Presque Isle,46.688958,-68.044797,America/New_York
1398,WFK,Northern Aroostook Regional Airport,47.285556,-68.312778,America/New_York


### Answer:

After filtering to U.S. airports (longitude < 0) and sorting by longitude in descending order (least negative value first), the easternmost airport in the United States is **Eastport Municipal Airport (EPM)** located in Eastport, Maine.

The airport has a longitude of -67.012694, which is the highest (closest to zero) longitude value among U.S. airports in the dataset.

### Validation & Assumptions:

Because all U.S. airports have negative longitude values (west of the Prime Meridian), the easternmost airport is the one with the largest longitude value (closest to zero).

The dataset result aligns with geographic knowledge that Eastport, Maine is the easternmost city in the continental United States. Therefore, Eastport Municipal Airport (EPM) is confirmed as the easternmost airport in the dataset.

# Question 3: Windiest New York Area Airport on February 12, 2013

In [9]:
# New York area airports
ny_airports = ["JFK", "LGA", "EWR"]

# Filter for February 12, 2013 and NY airports
w = weather[
    (weather["year"] == 2013) &
    (weather["month"] == 2) &
    (weather["day"] == 12) &
    (weather["origin"].isin(ny_airports))
].copy()

w.head()

,origin,year,month,day,hour,temp,dewp,humid,wind_dir,wind_speed,wind_gust,precip,pressure,visib,time_hour
1006,EWR,2013,2,12,0,39.92,39.02,96.55,240.0,6.90468,NaN,0.0,1006.9,10.0,2013-02-12T05:00:00Z
1007,EWR,2013,2,12,1,39.92,37.94,92.56,250.0,9.20624,NaN,0.0,1007.2,10.0,2013-02-12T06:00:00Z
1008,EWR,2013,2,12,2,39.92,28.04,62.21,270.0,20.71404,25.31716,0.0,1007.8,10.0,2013-02-12T07:00:00Z
1009,EWR,2013,2,12,3,39.02,26.96,61.63,260.0,1048.36058,NaN,0.0,1008.3,10.0,2013-02-12T08:00:00Z
1010,EWR,2013,2,12,4,39.02,26.96,64.29,280.0,12.65858,NaN,0.0,1008.8,10.0,2013-02-12T09:00:00Z


In [10]:
# Compare maximum wind speed and gust by airport
max_wind = (
    w.groupby("origin", as_index=False)
     .agg(
         max_wind_speed=("wind_speed", "max"),
         max_wind_gust=("wind_gust", "max"),
         avg_wind_speed=("wind_speed", "mean")
     )
     .sort_values("max_wind_speed", ascending=False)
)

max_wind

,origin,max_wind_speed,max_wind_gust,avg_wind_speed
0,EWR,1048.36058,31.07106,56.38822
2,LGA,23.01560,31.07106,14.96014
1,JFK,20.71404,27.61872,14.38475


In [11]:
# Remove unrealistic wind speeds (likely data errors)
w_clean = w[w["wind_speed"] < 200]

# Recalculate comparison
max_wind_clean = (
    w_clean.groupby("origin", as_index=False)
    .agg(
        max_wind_speed=("wind_speed", "max"),
        max_wind_gust=("wind_gust", "max"),
        avg_wind_speed=("wind_speed", "mean")
    )
    .sort_values("max_wind_speed", ascending=False)
)

max_wind_clean

,origin,max_wind_speed,max_wind_gust,avg_wind_speed
2,LGA,23.01560,31.07106,14.960140
0,EWR,21.86482,31.07106,13.258987
1,JFK,20.71404,27.61872,14.384750


### Answer:

After filtering the weather data for February 12, 2013 and limiting results to New York area airports (JFK, LGA, EWR), LaGuardia Airport (LGA) had the highest maximum wind speed at approximately 23.02 mph.

### Validation & Assumptions:

Initially, Newark (EWR) appeared to have an extremely high maximum wind speed (1048 mph), which is not physically possible. This value was identified as a clear data error or outlier.

To correct for this, wind speed values above 200 mph were removed from the dataset. After cleaning the data, LaGuardia Airport (LGA) was confirmed to have the highest recorded wind speed on February 12, 2013.

This demonstrates the importance of checking for outliers when analyzing real-world datasets.